# Cost & Latency Lab

Run the **same prompt** across four models — `gpt-5.5`, `gpt-5.5-instant`, `gpt-5.4-mini`, `gpt-5.4-nano` — at **three `reasoning_effort` levels** (`none`, `low`, `medium`). Collect **token counts** and **time-to-first-token (TTFT)**, then plot the trade-offs with matplotlib.

This makes the cost/latency/quality triangle concrete: pricier reasoning buys quality but costs tokens and latency.

> **Note on `gpt-5.5-instant`:** the "instant" variant is primarily a ChatGPT product concept; its dedicated API name was unconfirmed at authoring time. If the call errors with an unknown-model error, remove it from `MODELS` (its price is aliased to `gpt-5.5`'s below).
>
> **Cost note:** this sweep is 4 models × 3 efforts = **12 calls per prompt**, and we deliberately use a *substantive* prompt so the effort levels actually diverge. Expect real token usage; run it sparingly. Execution is deferred — it needs a valid API key.

## Setup

In [ ]:
import os, getpass, time
def _set_env(var):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")
_set_env("OPENAI_API_KEY")
# !pip install -U openai pandas matplotlib

from openai import OpenAI
import pandas as pd
client = OpenAI()

## 1. Define the sweep

In [ ]:
MODELS = [
    "gpt-5.5",
    "gpt-5.5-instant",   # remove if it errors as an unknown model name
    "gpt-5.4-mini",
    "gpt-5.4-nano",
]
EFFORTS = ["none", "low", "medium"]

# A substantive prompt so reasoning effort actually has something to chew on.
# (A trivial prompt collapses the effort levels together and hides the trade-off.)
PROMPT = (
    "Explain the CAP theorem with a concrete distributed systems example. "
    "Walk through what 'choosing' two of the three properties means in practice, "
    "and name a real datastore that sits on each side of the trade-off. Be thorough."
)

# Per-1M-token prices, standard tier (from research/01-model-variants.md).
# We track input AND output so we can estimate true per-call cost, not just output.
INPUT_PRICE_PER_1M = {
    "gpt-5.5": 5.00,
    "gpt-5.5-instant": 5.00,
    "gpt-5.4-mini": 0.75,
    "gpt-5.4-nano": 0.20,
}
OUTPUT_PRICE_PER_1M = {
    "gpt-5.5": 30.00,
    "gpt-5.5-instant": 30.00,
    "gpt-5.4-mini": 4.50,
    "gpt-5.4-nano": 1.25,
}

## 2. Run the sweep and measure TTFT + tokens

We stream each call so we can capture **time-to-first-token** (wall-clock from request to first streamed chunk), then read final token usage.

In [ ]:
def measure(model, effort, prompt):
    """Stream one call and capture latency + token usage.

    Returns a dict with:
      ttft_s       -- time to first streamed event (wall clock)
      total_s      -- total wall-clock time for the full response
      input_tokens, output_tokens, reasoning_tokens
    We stream so TTFT reflects real perceived latency, then read the final
    usage object off the terminal `response.completed` event.
    """
    start = time.perf_counter()
    ttft = None
    final_response = None

    stream = client.responses.create(
        model=model,
        input=prompt,
        reasoning={"effort": effort},
        stream=True,
    )

    for event in stream:
        # First event of ANY kind = first token of perceived latency.
        if ttft is None:
            ttft = time.perf_counter() - start
        # The completed event carries the final Response object (with usage).
        if getattr(event, "type", "") == "response.completed":
            final_response = event.response

    total = time.perf_counter() - start

    input_tokens = output_tokens = reasoning_tokens = None
    if final_response is not None and getattr(final_response, "usage", None):
        u = final_response.usage
        input_tokens = u.input_tokens
        output_tokens = u.output_tokens
        details = getattr(u, "output_tokens_details", None)
        reasoning_tokens = getattr(details, "reasoning_tokens", None) if details else None

    return {
        "ttft_s": ttft,
        "total_s": total,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "reasoning_tokens": reasoning_tokens,
    }


def estimate_cost(model, input_tokens, output_tokens):
    """True per-call cost: input tokens priced as input, output as output.
    (Reasoning tokens are already counted inside output_tokens by the API.)"""
    in_cost = (input_tokens or 0) / 1_000_000 * INPUT_PRICE_PER_1M.get(model, 0)
    out_cost = (output_tokens or 0) / 1_000_000 * OUTPUT_PRICE_PER_1M.get(model, 0)
    return in_cost + out_cost


rows = []
for model in MODELS:
    for effort in EFFORTS:
        try:
            m = measure(model, effort, PROMPT)
            cost = estimate_cost(model, m["input_tokens"], m["output_tokens"])
            rows.append({
                "model": model,
                "effort": effort,
                "ttft_s": round(m["ttft_s"] or 0, 3),
                "total_s": round(m["total_s"], 3),
                "input_tokens": m["input_tokens"],
                "output_tokens": m["output_tokens"],
                "reasoning_tokens": m["reasoning_tokens"],
                "est_cost_usd": round(cost, 6),
            })
        except Exception as e:
            rows.append({
                "model": model, "effort": effort,
                "ttft_s": None, "total_s": None,
                "input_tokens": None, "output_tokens": None,
                "reasoning_tokens": None, "est_cost_usd": None,
                "error": str(e),
            })

df = pd.DataFrame(rows)
df

## 3. Plot the trade-offs

Three panels, one per axis of the triangle: **latency** (TTFT), **output volume** (tokens), and **money** (estimated USD per call). Each line is one effort level swept across the four models.

In [ ]:
import matplotlib.pyplot as plt

plot_df = df.dropna(subset=["ttft_s", "output_tokens", "est_cost_usd"])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

panels = [
    ("ttft_s", "Time to First Token by Model", "TTFT (s)", "o"),
    ("output_tokens", "Output Tokens by Model", "Output tokens", "s"),
    ("est_cost_usd", "Estimated Cost per Call by Model", "USD per call", "^"),
]

for ax, (col, title, ylabel, marker) in zip(axes, panels):
    for effort in EFFORTS:
        sub = plot_df[plot_df["effort"] == effort]
        ax.plot(sub["model"], sub[col], marker=marker, label=f"effort={effort}")
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=30)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Reading the results

### How to read the three plots

- **Panel 1 (TTFT):** how long until the user sees *anything*. This is the number that governs perceived responsiveness in a chat UI. `none` effort should be flat and low; as effort climbs, the model spends time reasoning *before* the first visible token, so TTFT rises. For latency-critical surfaces (voice turns, autocomplete, classification) you want the bottom-left of this chart.
- **Panel 2 (output tokens):** how much the model actually wrote, including hidden reasoning tokens. Higher effort inflates this even when the visible answer is the same length — that's the reasoning tax. This is the main driver of cost on output-heavy work.
- **Panel 3 (USD per call):** the two above translated into money using the real per-token prices. The gap between `gpt-5.5` and `gpt-5.4-nano` here is the whole reason a model-routing strategy exists.

**The practical trade-off:** you are almost never choosing "the best model." You're choosing *the cheapest model at the lowest effort that still clears your quality bar* for a given task. A classification endpoint hit a million times a day belongs on `nano` at `none`; a one-shot legal analysis belongs on `gpt-5.5` at `high`. The plots tell you the price of each choice; only an eval (next notebook) tells you whether quality survives it.

The code cell below turns the per-call numbers into the decision you actually make: **monthly cost at scale.**

In [ ]:
# Turn per-call cost into a real budgeting decision.
# Scenario: a feature that makes this many calls per day.
CALLS_PER_DAY = 50_000
DAYS = 30

analysis = df.dropna(subset=["est_cost_usd"]).copy()
analysis["monthly_cost_usd"] = analysis["est_cost_usd"] * CALLS_PER_DAY * DAYS

print(f"Projected monthly cost at {CALLS_PER_DAY:,} calls/day for {DAYS} days:\n")
projection = (
    analysis
    .sort_values("monthly_cost_usd")
    [["model", "effort", "output_tokens", "ttft_s", "est_cost_usd", "monthly_cost_usd"]]
    .reset_index(drop=True)
)
projection["monthly_cost_usd"] = projection["monthly_cost_usd"].round(2)
print(projection.to_string(index=False))

# Headline comparison: cheapest vs most expensive configuration in the sweep.
cheapest = projection.iloc[0]
priciest = projection.iloc[-1]
ratio = priciest["monthly_cost_usd"] / max(cheapest["monthly_cost_usd"], 1e-9)
print(
    f"\nCheapest config: {cheapest['model']} @ effort={cheapest['effort']} "
    f"-> ${cheapest['monthly_cost_usd']:,.2f}/mo"
)
print(
    f"Priciest config: {priciest['model']} @ effort={priciest['effort']} "
    f"-> ${priciest['monthly_cost_usd']:,.2f}/mo"
)
print(f"\n==> Choosing the wrong model/effort here is a {ratio:,.0f}x cost difference.")

### Takeaways

- **Higher effort → more (reasoning) tokens and higher TTFT.** `none` is fastest and cheapest; `medium` is the balanced default for `gpt-5.5`. Don't pay for effort a task doesn't need.
- **The small models (`gpt-5.4-mini`, `gpt-5.4-nano`) are an order of magnitude cheaper per token.** For classification, extraction, and high-volume sub-agent calls, they're usually the right default.
- **Cost scales with traffic, not with the demo.** A few cents per call is invisible until you multiply by 50k calls/day — then the model/effort choice is the difference between a rounding error and a real line item.
- **Cost tells you the price; only an eval tells you if the cheap option is good enough.** Pair this lab with `mini-eval.ipynb`: find the cheapest config that still passes your quality bar, then pin it explicitly (model *and* `reasoning_effort`) so production cost is predictable.
- **Caveat:** these numbers are a single sample per cell. Latency varies run to run; for a real decision, average several runs and watch p95, not just the mean.